# Deep Learning with TensorFlow/Keras



---

## Learning Objectives

- Use TensorFlow/Keras Sequential API to build neural networks.
- Train an MNIST digit classifier with a Dense (MLP) network.
- Apply Dropout and L2 regularization to combat overfitting.
- Use EarlyStopping and ModelCheckpoint callbacks.
- Evaluate models with ROC curves and confusion matrices.
- Save and reload a trained model.

## Concept / Theory

### From Scikit-learn MLP to TensorFlow
Scikit-learn's MLPClassifier is great for tabular data but limited. TensorFlow/Keras gives full control over:
- Layer types (Dense, Conv2D, LSTM, etc.)
- Loss functions
- Optimizers (Adam, SGD, RMSprop)
- Custom callbacks

### Keras Sequential API
```python
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=20, validation_split=0.2)
```

### Overfitting & Regularization
- **Overfitting:** High training accuracy, low validation accuracy
- **Dropout:** Randomly sets neuron activations to 0 during training
- **L2 regularization:** Adds penalty for large weights

In [ ]:
# Setup
!pip install -q tensorflow scikit-learn matplotlib seaborn pandas numpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## Practical Lab 1 – MNIST Digit Classification (Dense NN)

### Objective
Build, train, and evaluate a Dense neural network on the MNIST handwritten digit dataset.

### Explanation
MNIST contains 70,000 28×28 grayscale images of digits 0-9. We flatten each image to a 784-element vector for a Dense network.

In [ ]:
# Lab 1: Load & prepare MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

print(f'Training:   {x_train.shape}')
print(f'Test:       {x_test.shape}')
print(f'Classes:    {np.unique(y_train)}')

# Show sample images
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap='gray_r')
    ax.set_title(str(y_train[i]), fontsize=9)
    ax.axis('off')
plt.suptitle('MNIST Sample Images'); plt.tight_layout(); plt.show()

In [ ]:
# Build Dense neural network
model = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten(),                        # 28×28 → 784
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10,  activation='softmax')  # 10 digit classes
], name='MNIST_Dense')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# Train
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

# Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'\nTest Accuracy: {test_acc:.4f}')
print(f'Test Loss:     {test_loss:.4f}')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric, title in zip(axes, ['accuracy','loss'], ['Accuracy','Loss']):
    ax.plot(history.history[metric], lw=2, label='Train')
    ax.plot(history.history[f'val_{metric}'], lw=2, label='Validation', linestyle='--')
    ax.set_title(f'MNIST {title}')
    ax.set_xlabel('Epoch'); ax.set_ylabel(metric.capitalize())
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Expected Output
```
Test Accuracy: ~0.978
```

## Practical Lab 2 – MLP for Breast Cancer (Tabular Data)

### Objective
Build a Keras network for binary tabular classification.

### Explanation
Binary classification uses `sigmoid` output and `binary_crossentropy` loss (not softmax/sparse_categorical).

In [ ]:
# Lab 2: Breast Cancer dataset
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, stratify=y_c, random_state=42)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

# Build model for binary classification
clf_model = keras.Sequential([
    keras.layers.Input(shape=(X_c.shape[1],)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1,  activation='sigmoid')  # binary output
])

clf_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

clf_history = clf_model.fit(
    X_tr_sc, y_tr,
    epochs=50, batch_size=16,
    validation_split=0.15,
    verbose=0
)

test_loss, test_acc = clf_model.evaluate(X_te_sc, y_te, verbose=0)
print(f'Breast Cancer Test Accuracy: {test_acc:.4f}')

## Practical Lab 3 – Overfitting: Dropout & L2 Regularization

### Objective
Detect overfitting and apply regularization techniques.

### Explanation
- **Overfitting sign:** val_loss increases while train_loss decreases.
- **Dropout(rate):** During training, randomly zero out `rate` fraction of neurons. Forces redundant representations.
- **L2 Regularization:** Adds λ * ||w||² to the loss, penalizing large weights.

In [ ]:
# Lab 3: Build model WITH regularization
from tensorflow.keras.regularizers import l2

reg_model = keras.Sequential([
    keras.layers.Input(shape=(X_c.shape[1],)),
    keras.layers.Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation='sigmoid')
])
reg_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
reg_history = reg_model.fit(X_tr_sc, y_tr, epochs=80, batch_size=16,
                             validation_split=0.15, verbose=0)

reg_loss, reg_acc = reg_model.evaluate(X_te_sc, y_te, verbose=0)
print(f'Regularized model test accuracy: {reg_acc:.4f}')

In [ ]:
# Compare baseline vs regularized
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for i, (hist, label) in enumerate([(clf_history,'Baseline'),(reg_history,'Regularized')]):
    axes[i].plot(hist.history['loss'], label='Train Loss')
    axes[i].plot(hist.history['val_loss'], label='Val Loss', linestyle='--')
    axes[i].set_title(f'{label} Model — Loss')
    axes[i].set_xlabel('Epoch'); axes[i].legend(); axes[i].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('Tip: If val_loss diverges from train_loss, you have overfitting!')

## Practical Lab 4 – EarlyStopping & ModelCheckpoint

### Objective
Use callbacks to stop training at the right time and save the best model.

### Explanation
- **EarlyStopping:** Monitors a metric; stops if no improvement after `patience` epochs. Restores best weights.
- **ModelCheckpoint:** Saves the model whenever the monitored metric improves.

In [ ]:
# Lab 4: Callbacks
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)
checkpoint = keras.callbacks.ModelCheckpoint(
    filepath='best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=0
)

# Rebuild model
cb_model = keras.Sequential([
    keras.layers.Input(shape=(X_c.shape[1],)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])
cb_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

cb_history = cb_model.fit(
    X_tr_sc, y_tr,
    epochs=200,
    batch_size=16,
    validation_split=0.15,
    callbacks=[early_stop, checkpoint],
    verbose=0
)
print(f'Stopped at epoch: {len(cb_history.history["loss"])}')

# Load best model
best = keras.models.load_model('best_model.keras')
best_acc = best.evaluate(X_te_sc, y_te, verbose=0)[1]
print(f'Best model test accuracy: {best_acc:.4f}')

## Practical Lab 5 – Evaluation: ROC, Confusion Matrix

### Objective
Go beyond accuracy — use ROC curve, AUC, and confusion matrix.

### Explanation
- **ROC curve** plots True Positive Rate vs False Positive Rate at different thresholds.
- **AUC** (Area Under Curve) — 1.0 is perfect, 0.5 is random.
- **Confusion matrix** shows exact counts of TP, TN, FP, FN.

In [ ]:
# Lab 5: Detailed evaluation
y_prob = cb_model.predict(X_te_sc, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

print('Classification Report:')
print(classification_report(y_te, y_pred, target_names=['Malignant','Benign']))

# ROC curve
fpr, tpr, _ = roc_curve(y_te, y_prob)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr, tpr, lw=2, label=f'ROC (AUC={roc_auc:.3f})')
axes[0].plot([0,1],[0,1],'k--', label='Random')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve — Breast Cancer')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Confusion matrix
cm = confusion_matrix(y_te, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Malignant','Benign'], yticklabels=['Malignant','Benign'])
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix')
plt.tight_layout(); plt.show()

In [ ]:
# Save and reload
cb_model.save('day8_cancer_model.keras')
loaded = keras.models.load_model('day8_cancer_model.keras')
print('Model saved and reloaded successfully!')
print('Reloaded accuracy:', loaded.evaluate(X_te_sc, y_te, verbose=0)[1])

### Expected Output
```
ROC AUC: ~0.99
Test Accuracy: ~0.97+
```

## Exercises

**Exercise 1:** Add a third hidden layer (64 neurons, ReLU) to the MNIST model. Compare accuracy.

In [ ]:
# Exercise 1 — Modify MNIST model with extra layer

**Exercise 2:** Change the Dropout rate to 0.5 and observe how it affects training/validation accuracy.

In [ ]:
# Exercise 2 — Test Dropout(0.5) on Breast Cancer model

**Exercise 3:** Use `model.predict()` on 5 MNIST test samples. Show image, true label, and predicted probabilities.

In [ ]:
# Exercise 3 — Predict on MNIST samples and visualize

## Challenge Task – Tune MNIST to >98%

Improve the MNIST model to exceed 98% test accuracy:
1. Try BatchNormalization between Dense layers
2. Use learning rate scheduling
3. Train for more epochs with EarlyStopping
4. Try Adam with different learning rates (1e-3, 1e-4)
5. Report your best configuration and accuracy

In [ ]:
# Challenge: Reach >98% on MNIST
from tensorflow.keras.callbacks import ReduceLROnPlateau

best_mnist = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(10, activation='softmax')
])

best_mnist.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

best_hist = best_mnist.fit(
    x_train, y_train,
    epochs=30,
    batch_size=128,
    validation_split=0.1,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
    ],
    verbose=1
)

best_acc = best_mnist.evaluate(x_test, y_test, verbose=0)[1]
print(f'Best MNIST accuracy: {best_acc:.4f}')
if best_acc >= 0.98:
    print('✓ Achieved >98% target!')

## Real-World AI/ML Application

| Deep Learning Tool | Application |
|---|---|
| **Dense Network** | Fraud detection, credit scoring, medical diagnosis |
| **Dropout** | Used in almost every production model to prevent overfitting |
| **EarlyStopping** | Saves training time, prevents memorization |
| **ModelCheckpoint** | Production pipeline — save best model during training |
| **ROC/AUC** | Medical screening, fraud detection — where threshold matters |

## Day Summary

| Concept | Key Point |
|---|---|
| Keras Sequential | layer-by-layer model definition |
| `model.compile()` | optimizer, loss, metrics |
| `model.fit()` | training loop |
| `validation_split` | holdout for early stopping |
| **Dropout** | zero neurons randomly → generalization |
| **L2 regularizer** | penalize large weights |
| **EarlyStopping** | stop at optimal epoch |
| **ROC/AUC** | evaluation beyond accuracy |

**→ Day 9: Computer Vision & Image Processing**